In [4]:
import ftd2xx
print(ftd2xx.listDevices())

[b'']


In [2]:
import ftd2xx
import time

class Gilson4020PumpUSB:
    def __init__(self, ftdi_serial=None):
        devices = ftd2xx.listDevices()
        if not devices:
            raise RuntimeError("No FTDI devices found.")

        decoded_devices = [d.decode() if isinstance(d, bytes) else d for d in devices]
        if ftdi_serial is None:
            ftdi_serial = decoded_devices[0]  # Use first available device
        elif ftdi_serial not in decoded_devices:
            raise RuntimeError(f"FTDI device with serial '{ftdi_serial}' not found. Available: {decoded_devices}")

        try:
            self.device = ftd2xx.openEx(ftdi_serial.encode() if isinstance(ftdi_serial, str) else ftdi_serial, 1)
        except ftd2xx.DeviceError as e:
            raise RuntimeError(f"Failed to open FTDI device '{ftdi_serial}': {e}")

        self.device.setTimeouts(1000, 1000)
        self.device.purge()
        self.device.setDtr()  # enable DTR
        self.device.setRts()  # enable RTS
        self.device.setLatencyTimer(16)
        self.device.setUSBParameters(4096, 4096)
        self.device.setChars(0, False, 0, False)
        # self.device.setEventNotification(0)  # Removed due to missing required argument
        self.device.setFlowControl(0x0100, 0x11, 0x13)
        self.device.setBitMode(0, 0x00)  # Reset
        self.device.setBitMode(0, 0x40)  # MPSSE mode
        self.device.setEcho(False)  # Disable echo

    def send_command(self, cmd):
        self.device.write((cmd + "\r").encode('ascii'))
        time.sleep(0.5)
        response = self.device.read(1024)
        return response.decode('ascii', errors='ignore').strip()

    def identify(self):
        return self.send_command('%')

    def home(self):
        return self.send_command('p')

    def get_serial_number(self):
        self.send_command('.sn')
        time.sleep(0.2)
        return self.send_command('.')

    def reset(self):
        return self.send_command('$')

    def close(self):
        self.device.close()

In [3]:
pump = Gilson4020PumpUSB()
print("Identify:", pump.identify())
print("Serial:", pump.get_serial_number())
print("Home:", pump.home())
pump.close()


AttributeError: 'FTD2XX' object has no attribute 'setEcho'